# Recovery with Gallagher et al.'s full-history routine

- **Patients:** 1000, each with freshly drawn parameters, the way their DRL repo draws virtual patients (`DRL_Personalized_AT/demo/patients/generation.py`: uniform draws, redrawn until the patient progresses within 5000 days). turnover ~ U(0.05, 0.6), K ~ U(2.5, 5) and noise SD ~ U(0.065, 0.6) span the range of their first-cycle Bruchovsky fits; cost 0.1, fR 1e-5, rS 0.027, dD 1.5 and n0 = 1 are fixed as in those fits. Each is simulated under adaptive therapy with their `Simulate_AT` (Zhang rule, 50% threshold, 30-day visits); PSA is divided by the first value, Gaussian noise is added to every visit after the first, and the record stops at 1.2 times baseline.
- **Fit:** their full-history routine, `Fig2_Probing_Cycle.ipynb` cells 9-10. Cell 9 (settings) is run unchanged from their notebook; cell 10's fit call is copied. Six free parameters (n0, fR, cost, turnover, K, rS), dD fixed at 1.5, one fit per patient.
- **Needs** `lmfit` and their repo cloned at `../../mathematical_biomarkers`.

In [ ]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd
from multiprocessing import get_context
from lmfit import minimize, Parameters
warnings.filterwarnings('ignore')

repo = os.path.abspath('../../mathematical_biomarkers')
sys.path.insert(0, os.path.join(repo, 'utils'))
from odeModels import LotkaVolterraModel
from fittingUtils import residual

n_patients, seed = 1000, 1
names = ['n0', 'fR', 'cost', 'turnover', 'K', 'rS']
out_dir = '../synth_data_gallagher'
os.makedirs(out_dir, exist_ok=True)

In [ ]:
rng = np.random.default_rng(seed)
fixed = {'cost': 0.1, 'fR': 1e-5, 'rS': 0.027, 'dD': 1.5, 'n0': 1}

def draw_patient(pid):
    #redraw until the patient progresses within 5000 days, as in their DRL generation.py
    while True:
        p = {**fixed, 'turnover': rng.uniform(0.05, 0.6), 'K': rng.uniform(2.5, 5), 'noise_sd': rng.uniform(0.065, 0.6), 'PatientId': pid}
        p.update(rR=(1 - p['cost']) * p['rS'], dS=p['turnover'] * p['rS'], dR=p['turnover'] * p['rS'], S0=p['n0'] * (1 - p['fR']), R0=p['n0'] * p['fR'])
        model = LotkaVolterraModel(dt=1)
        model.SetParams(**p)
        model.Simulate_AT(atThreshold=0.5, intervalLength=30, t_end=5000)
        if np.isfinite(model.ttp):
            break
    df = model.resultsDf.drop_duplicates('Time', keep='last')
    df = df[df.TumourSize <= 1.2 * df.TumourSize.iloc[0]]
    df = df[df.Time % 30 == 0].reset_index(drop=True)
    noise = rng.normal(0, p['noise_sd'], len(df))
    noise[0] = 0
    return p, df.assign(PSA=df.TumourSize / df.TumourSize.iloc[0] + noise, PatientId=pid)[['PatientId', 'Time', 'DrugConcentration', 'PSA']]

draws = [draw_patient(pid) for pid in range(n_patients)]
truth = pd.DataFrame([p for p, _ in draws])
patients = pd.concat([d for _, d in draws], ignore_index=True)
truth.to_csv(os.path.join(out_dir, 'true_params.csv'), index=False)
patients.groupby('PatientId').size().describe()

In [ ]:
#their settings: cell 9 of Fig2_Probing_Cycle.ipynb, run as it is
cell9 = ''.join(json.load(open(os.path.join(repo, 'paper_figures/Fig2_Probing_Cycle.ipynb')))['cells'][9]['source'])

def fit_patient(training_data_df):
    ns = {'Parameters': Parameters, 'os': os, 'out_dir': out_dir, 'patientIds': []}
    exec(cell9, ns)
    params, solver_kws, optimiser_kws = ns['params'], ns['solver_kws'], ns['optimiser_kws']
    #cell 10
    tmp_model = LotkaVolterraModel()
    tmp_model.SetParams(**params)
    fit_obj = minimize(residual, params, args=(training_data_df.reset_index(drop=True), tmp_model, "PSA", solver_kws), **optimiser_kws)
    at_bound = {f'{k}_at_bound': bool(np.isclose(fit_obj.params[k].value, [fit_obj.params[k].min, fit_obj.params[k].max], rtol=1e-4, atol=1e-8).any()) for k in names}
    return {'PatientId': training_data_df.PatientId.iloc[0], 'converged': fit_obj.success, **{k: fit_obj.params[k].value for k in names}, **at_bound}

t = time.time()
with get_context('fork').Pool(8) as pool:
    est = pd.DataFrame(pool.map(fit_patient, [d for _, d in patients.groupby('PatientId')]))
est.to_csv(os.path.join(out_dir, 'fits.csv'), index=False)
print(f'{len(est)} fits in {(time.time() - t) / 60:.0f} min, {int(est.converged.sum())} converged')

In [ ]:
m = est.merge(truth[['PatientId', *names]], on='PatientId', suffixes=('', '_true'))
pd.DataFrame({k: {'truth': f"{m[k + '_true'].min():.3g}-{m[k + '_true'].max():.3g}",
                  'bias': (m[k] - m[k + '_true']).mean(),
                  'RMSE': np.sqrt(((m[k] - m[k + '_true']) ** 2).mean()),
                  'corr': m[k].corr(m[k + '_true']),
                  'at bound': m[k + '_at_bound'].mean()} for k in names}).T